# Visual AU Extraction — MELD (OpenFace 2.0 fallback)

Fallback AU pipeline using OpenFace 2.0 CLI binary.
Run this instead of Section 3 in `extract_video_meld.ipynb` when OpenFace 3.0 is unavailable.

**Prerequisite:** Compile OpenFace 2.0 and set `OPENFACE2_BIN` in the config cell.
```bash
sudo apt-get install cmake libopenblas-dev libopencv-dev libdlib-dev
git clone https://github.com/TadasBaltrusaitis/OpenFace.git
cd OpenFace && mkdir build && cd build
cmake -D CMAKE_BUILD_TYPE=Release .. && make -j$(nproc)
# Binary at: OpenFace/build/bin/FeatureExtraction
```

Outputs: `Dataset/Processed/MELD/features/video_openface2_au_{split}.pt` — `{clip_name: np.array(18,)}`

In [ ]:
import os, sys, subprocess, tempfile, torch, numpy as np, pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

PROJECT_ROOT = Path('/mnt/Work/ML/Thesis/BMVC/Hopeful')
DATA_ROOT    = PROJECT_ROOT / 'Dataset/Processed/MELD'
FEATURES_DIR = DATA_ROOT / 'features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

OPENFACE2_BIN     = Path('/path/to/OpenFace/build/bin/FeatureExtraction')  # EDIT THIS
AU_CONF_THRESHOLD = 0.9

print(f'OpenFace2 bin: {OPENFACE2_BIN}')
print(f'Exists:        {OPENFACE2_BIN.exists()}')

In [ ]:
labels = pd.read_csv(DATA_ROOT / 'labels.csv')
SPLITS = ['train', 'dev', 'test']
for split in SPLITS:
    n = len(labels[labels['split'] == split])
    print(f'  {split}: {n} utterances')

In [ ]:
AU_COLS = [
    'AU01_r', 'AU02_r', 'AU04_r', 'AU05_r', 'AU06_r', 'AU07_r',
    'AU09_r', 'AU10_r', 'AU12_r', 'AU14_r', 'AU15_r', 'AU17_r',
    'AU20_r', 'AU23_r', 'AU25_r', 'AU26_r', 'AU28_r', 'AU45_r',
]
AU_DIM = len(AU_COLS)  # 18
print(f'AU dim: {AU_DIM}')

def extract_au_of2(video_path: Path):
    with tempfile.TemporaryDirectory() as tmpdir:
        subprocess.run(
            [str(OPENFACE2_BIN), '-f', str(video_path), '-out_dir', tmpdir, '-aus'],
            capture_output=True, timeout=300,
        )
        csv_files = list(Path(tmpdir).glob('*.csv'))
        if not csv_files:
            return None
        df = pd.read_csv(csv_files[0])
        df.columns = df.columns.str.strip()
        if 'confidence' in df.columns:
            df = df[df['confidence'] >= AU_CONF_THRESHOLD]
        if 'success' in df.columns:
            df = df[df['success'] == 1]
        if len(df) == 0:
            return None
        vec = np.zeros(AU_DIM, dtype=np.float32)
        for i, col in enumerate(AU_COLS):
            if col in df.columns:
                vec[i] = float(df[col].mean())
        return vec

In [ ]:
assert OPENFACE2_BIN.exists(), f'Binary not found: {OPENFACE2_BIN}'

for split in SPLITS:
    out_path = FEATURES_DIR / f'video_openface2_au_{split}.pt'
    if out_path.exists():
        print(f'{split}: already exists — skipping'); continue
    subset    = labels[labels['split'] == split]
    features  = {}
    skipped   = []
    zero_face = []

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=f'AU OF2 {split}'):
        vid_path = DATA_ROOT / row['video_path']
        if not vid_path.exists():
            skipped.append(row['clip_name']); continue
        au_vec = extract_au_of2(vid_path)
        if au_vec is None:
            zero_face.append(row['clip_name'])
            features[row['clip_name']] = np.zeros(AU_DIM, dtype=np.float32)
            continue
        features[row['clip_name']] = au_vec

    torch.save(features, out_path)
    cov = len(features) - len(zero_face)
    print(f'Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)')
    print(f'Face coverage: {cov}/{len(features)}  ({cov/max(len(features),1)*100:.1f}%)')
    if skipped:   print(f'Skipped (missing): {skipped}')
    if zero_face: print(f'No face detected:  {len(zero_face)} clips')

## Verification

In [ ]:
EXPECTED = {s: len(labels[labels['split'] == s]) for s in SPLITS}
print('=== Verification ===')
for split in SPLITS:
    pt = FEATURES_DIR / f'video_openface2_au_{split}.pt'
    if not pt.exists(): print(f'  {split}: MISSING'); continue
    d = torch.load(pt, weights_only=False)
    all_feats = np.stack(list(d.values()))
    print(f'  {split}: count={len(d)}/{EXPECTED[split]}  shape={next(iter(d.values())).shape}  '
          f'nan={np.isnan(all_feats).any()}  inf={np.isinf(all_feats).any()}')